In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/players_features.csv")
print(df.shape)

(18404, 25)


In [2]:
y = np.log1p(df["market_value_in_eur"])
X = df.drop(columns=["player_id", "name", "market_value_in_eur"])
X.columns.tolist()

['country_of_citizenship',
 'sub_position',
 'position',
 'foot',
 'height_in_cm',
 'international_caps',
 'international_goals',
 'current_club_domestic_competition_id',
 'caps_missing',
 'age',
 'contract_months_left',
 'no_league',
 'has_contract_info',
 'games',
 'minutes',
 'goals',
 'assists',
 'yellow',
 'red',
 'goals_per90',
 'assists_per90',
 'has_recent_games']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

(14723, 22) (3681, 22)


In [4]:
num_cols = ["age", "height_in_cm", "international_caps", "international_goals",
            "contract_months_left", "games", "minutes", "goals", "assists",
            "yellow", "goals_per90", "assists_per90"]
cat_cols = ["country_of_citizenship", "position", "sub_position",
            "foot", "current_club_domestic_competition_id"]
bin_cols = ["caps_missing", "no_league", "has_contract_info", "has_recent_games"]

In [5]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30,
                          sparse_output=False), cat_cols),
    ("bin", "passthrough", bin_cols),
])

In [6]:
results = []

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    mae_eur = mean_absolute_error(np.expm1(y_test), np.expm1(pred))
    results.append({"model": name, "RMSE_log": rmse, "MAE_log": mae,
                    "R2_log": r2, "MAE_EUR": mae_eur})
    return pd.DataFrame(results).round(3)

In [7]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingRegressor

boost = Pipeline([("prep", preprocessor),
                  ("model", GradientBoostingRegressor(random_state=42))])

scores = cross_val_score(boost, X_train, y_train, cv=5, scoring="r2")
print(scores)
print(scores.mean())

[0.6811472  0.68275121 0.676699   0.71305926 0.68556979]
0.6878452910235832


In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [3, 4],
    "model__learning_rate": [0.05, 0.1],
}
grid = GridSearchCV(boost, param_grid, cv=3, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

{'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 300}
0.724709796550084


In [9]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__n_estimators": [300, 400, 500],
    "model__max_depth": [4, 5, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
}
search = RandomizedSearchCV(boost, param_dist, n_iter=10, cv=3,
                            scoring="r2", n_jobs=-1, random_state=42)
search.fit(X_train, y_train)
print(search.best_params_)
print(search.best_score_)

{'model__subsample': 0.8, 'model__n_estimators': 500, 'model__max_depth': 6, 'model__learning_rate': 0.05}
0.7298674838229612


In [10]:
evaluate("Gradient Boosting (default)", boost)
best_model = search.best_estimator_
evaluate("Tuned Gradient Boosting", best_model)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Gradient Boosting (default),0.898,0.696,0.684,1713087.269
1,Tuned Gradient Boosting,0.820,0.624,0.736,1417827.480


In [11]:
import joblib

joblib.dump(best_model, "../models/best_model.joblib")

['../models/best_model.joblib']